In [ ]:
from marslab.imgops.look import save_plainly
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.ndimage import gaussian_filter

from asdf.format import compile_looks
from asdf_settings.generators import make_bilateralfilter

from rapid.algebra import make_classifier_look
from rapid.helpers import get_zcam_bandset

mpl.rcParams['image.interpolation'] = None
mpl.rcParams['figure.dpi'] = 250
rapidlooks = rapidlooks = {
    look['name']: look for look in compile_looks()
}

# I like to see popup plots in separate windows I can maximize.
# if you don't want that, replace 'qt' with 'notebook'.
%matplotlib qt

In [ ]:
image_path = '/home/michael/Desktop/zcam_data/products/0130/iof/'
roi_path = None
# observation is a ZcamBandSet object -- a subclass of our multispectral 
# data multitool/abstraction with special behavior for ZCAM.
observation = get_zcam_bandset(image_path, roi_path)

In [ ]:
# the observation object knows lots of useful things about ZCAM in general 
# and these images in particular.
observation.wavelength('R1'), observation.summary['SOLAR_ELEVATION']

the syntax for defining spectrum operations is fairly straightforward. notes: 
* for band depth, bands are ordered (left, right, center). 
* allowable values for 'look' are: band_depth, ratio, slope, band_avg (average reflectance across all bands between first and second band inclusive), band_min (wavelength of minimum-valued band between first and second band inclusive), band_max (wavelength of maximum-valued band between first and second band inclusive).

In [ ]:
BD866 = {'look': 'band_depth', 'bands': ('R1', 'R6', 'R2')}
BD910 = {'look': 'band_depth', 'bands': ('R1', 'R5', 'R3')}
BD939 = {'look': 'band_depth', 'bands': ('R1', 'R6', 'R4')}
BD978 = {'look': 'band_depth', 'bands': ('R1', 'R6', 'R5')}
R800_1022 = {'look': 'ratio', 'bands': ('R1', 'R6')}
R631_800 = {'look': 'ratio', 'bands': ('R0R', 'R1')}
R800_631 = {'look': 'ratio', 'bands': ('R1', 'R0R')}

syntax for these class definitions is also pretty straightforward. notes:  
* currently this implicitly places a boolean AND between each condition of a class-membership definition. 
* if more sophisticated control over the logic of either of these behaviors is desired, we can implement it.
* class definitions flow top-to-bottom exclusive (like IDL CASE, as opposed to IDL SWITCH) -- a pixel is colored in the consolidated classification map according to the first class it falls into. 

* any [standard CSS color](https://www.w3.org/wiki/CSS/Properties/color/keywords) should be a legal value for 'color'.

In [ ]:
spectral_class_definitions = {
    'hematite': {
        'conditions': (
            BD866 | {'range': (0.08, 0.2)},
            R800_631 | {'range': (1.2, 2.0)}
        ),
        'color': 'firebrick'
    },
    'red slope':  {
        'conditions': (
            R631_800 | {'range': (0.4, 0.8)},
            R800_1022 | {'range': (0.6, 0.9)}
        ),
        'color': 'lightcoral'
    },
    'blue slope': {
        'conditions': (
            R800_1022 | {'range': (1.1, 1.4)},
            R631_800 | {'range': (1.0, 1.6)}
        ),
        'color': 'blue'
    },
    '>900nm band': {
        'conditions': (
            BD939 | {'range': (0.03, 0.2)},
            BD978 | {'range': (0.05, 0.2)}
        ),
        'color': 'darkorchid'
    },
    '900nm band': {
        'conditions': (
            BD910 | {'range': (0.03, 0.2)},
            BD939 | {'range': (0.03, 0.2)}
        ),
        'color': 'darkgreen'
    },
}

In [ ]:
# a recommended prefilter. consider playing with the sigma parameter.
prefilter = {"function": gaussian_filter, "params": {"sigma": 1}}
# a different filter that's sensitive to edges.
# prefilter = {"function": make_bilateralfilter(15, 3, 7)}

# this is 'experimental' functionality, so it has its very own assembler.
rclook = make_classifier_look(
    spectral_class_definitions,
    observation,
    # if you don't want to use a filter, just set this to None.
    prefilter = prefilter,
    # this parameter renders maps for each class as well as the consolidated map
    plot_all = True  
)
class_plots = rclook.execute()

In [ ]:
# you can also render the standard asdf rapidlooks from this notebook, 
# although font sizes etc. may not be beautifully perfect
observation.make_look_set(
    [rapidlooks['slope R1_R6'], rapidlooks['enhanced color L0R_L0G_L0B']]
)

In [ ]:
# save a look -- again, maybe mess with font sizes, WIP, etc.
save_plainly(class_plots[0], '.', 'spectral_class_map.png', dpi=150)

In [ ]:
# run to close all matplotlib figures
plt.close('all')